# Data Preparation

This notebook acts as the foundational step for the telemetry analysis pipeline. It is responsible for loading the selected team and Grand Prix data, fetching both drivers' session information, and executing a robust cleaning and interpolation workflow. The resulting processed telemetry is then cached for efficient reuse across subsequent analytical and visualization notebooks.

## Setup and Dependencies


Import the necessary external libraries (such as fastf1) and initialize your internal project modules located in the src directory. This section also sets the project_root to ensure consistent file path resolution across the entire repository.

In [1]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.config import *
from src.load_save_data import *
from src.preprocess_data import *

## Get Drivers Telemetry

Identify the two teammates for the selected team and session. This step retrieves the raw lap and telemetry datasets from the fastf1 API. The raw data is briefly inspected before being stored in the cache for preliminary record-keeping.

In [2]:
session = fastf1.get_session(YEAR, GRAND_PRIX, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, TEAM)
segment = get_segment(session, driver_1, driver_2)

lap_1, lap_2 = load_teammates_data(YEAR, GRAND_PRIX, segment, driver_1, driver_2)

print(f"\nDrivers names: {lap_1['Driver']}, {lap_2['Driver']}")
print(f"Lap times: {lap_1['LapTime']}, {lap_2['LapTime']}")
print(f"Lap start times: {lap_1['LapStartTime']}, {lap_2['LapStartTime']}")
print(f"Tyre compounds: {lap_1['TyreCompound']}, {lap_2['TyreCompound']}")
print(f"Tyre ages: {lap_1['TyreAge']}, {lap_2['TyreAge']}", "\n")
print(f"{lap_1['Driver']} telemetry data info:\n")
display(lap_1['Telemetry'].info())
print(f"{lap_1['Driver']} telemetry data head:")
display(lap_1['Telemetry'].head())
print(f"{lap_2['Driver']} telemetry data info:\n")
display(lap_2['Telemetry'].info())
print(f"{lap_2['Driver']} telemetry data head:")
display(lap_2['Telemetry'].head())



req         WARNING 	DEFAULT CACHE ENABLED! (99.92 MB) C:\Users\joseg\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '6', '44', '23', '87', '10', '55', '14', '30', '22', '27', '5', '31', '7', '18']
core         


Drivers names: ALO, STR
Lap times: 0 days 00:01:28.337000, 0 days 00:01:29.271000
Lap start times: 0 days 00:31:22.302000, 0 days 00:20:37.637000
Tyre compounds: SOFT, SOFT
Tyre ages: 2.0, 2.0 

ALO telemetry data info:

<class 'fastf1.core.Telemetry'>
Index: 669 entries, 2 to 670
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype          
---  ------                 --------------  -----          
 0   Date                   669 non-null    datetime64[ns] 
 1   SessionTime            669 non-null    timedelta64[ns]
 2   DriverAhead            669 non-null    object         
 3   DistanceToDriverAhead  669 non-null    float64        
 4   Time                   669 non-null    timedelta64[ns]
 5   RPM                    669 non-null    float64        
 6   Speed                  669 non-null    float64        
 7   nGear                  669 non-null    int64          
 8   Throttle               669 non-null    float64        
 9   Brake              

None

ALO telemetry data head:


,Date,SessionTime,DriverAhead,DistanceToDriverAhead,Time,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Distance,RelativeDistance,Status,X,Y,Z
2,2025-04-05 06:16:15.845,0 days 00:29:53.965000,,792.985278,0 days 00:00:00,11407.714286,277.371429,7,99.0,False,12,interpolation,0.103784,0.000018,OnTrack,1624.204764,-560.273551,743.116330
3,2025-04-05 06:16:16.029,0 days 00:29:54.149000,,792.985278,0 days 00:00:00.184000,11549.000000,280.000000,7,99.0,False,12,car,14.417778,0.002490,OnTrack,1716.843499,-669.174928,739.148428
4,2025-04-05 06:16:16.121,0 days 00:29:54.241000,,792.985278,0 days 00:00:00.276000,11600.060000,280.460000,7,99.0,False,12,pos,21.579261,0.003727,OnTrack,1763.000000,-723.000000,737.000000
5,2025-04-05 06:16:16.229,0 days 00:29:54.349000,18,792.985278,0 days 00:00:00.384000,11660.000000,281.000000,7,99.0,False,12,car,30.028889,0.005186,OnTrack,1818.299860,-787.203796,734.316939
6,2025-04-05 06:16:16.241,0 days 00:29:54.361000,18,792.605278,0 days 00:00:00.396000,11664.200000,281.250000,7,99.0,False,12,pos,30.973564,0.005349,OnTrack,1825.000000,-795.000000,734.000000


STR telemetry data info:

<class 'fastf1.core.Telemetry'>
Index: 675 entries, 2 to 676
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype          
---  ------                 --------------  -----          
 0   Date                   675 non-null    datetime64[ns] 
 1   SessionTime            675 non-null    timedelta64[ns]
 2   DriverAhead            675 non-null    object         
 3   DistanceToDriverAhead  675 non-null    float64        
 4   Time                   675 non-null    timedelta64[ns]
 5   RPM                    675 non-null    float64        
 6   Speed                  675 non-null    float64        
 7   nGear                  675 non-null    int64          
 8   Throttle               675 non-null    float64        
 9   Brake                  675 non-null    bool           
 10  DRS                    675 non-null    int64          
 11  Source                 675 non-null    object         
 12  Distance               675 non-null

None

STR telemetry data head:


,Date,SessionTime,DriverAhead,DistanceToDriverAhead,Time,RPM,Speed,nGear,Throttle,Brake,DRS,Source,Distance,RelativeDistance,Status,X,Y,Z
2,2025-04-05 06:05:30.246,0 days 00:19:08.366000,,86.009167,0 days 00:00:00,11263.844359,274.938637,7,99.432954,False,12,interpolation,1.840340,0.000318,OnTrack,1616.763413,-553.159249,743.04837
3,2025-04-05 06:05:30.627,0 days 00:19:08.747000,,86.009167,0 days 00:00:00.381000,11505.000000,281.000000,7,99.000000,False,12,car,31.679722,0.005467,OnTrack,1812.730356,-781.556307,734.32870
4,2025-04-05 06:05:30.641,0 days 00:19:08.761000,,86.009167,0 days 00:00:00.395000,11512.388821,281.116666,7,99.038889,False,12,pos,32.779083,0.005657,OnTrack,1820.000000,-790.000000,734.00000
5,2025-04-05 06:05:30.941,0 days 00:19:09.061000,,86.009167,0 days 00:00:00.695000,11670.722155,283.616666,7,99.872222,False,12,pos,56.422592,0.009738,OnTrack,1974.000000,-969.000000,727.00000
6,2025-04-05 06:05:30.987,0 days 00:19:09.107000,31,86.009167,0 days 00:00:00.741000,11695.000000,284.000000,7,100.000000,False,12,car,60.079722,0.010369,OnTrack,1996.176229,-995.037275,726.00525


## Data Preprocessing

Standardize and clean the raw telemetry data. This includes handling data inconsistencies, formatting timestamps, and ensuring that all telemetry channels are prepared for comparative analysis. The cleaned outputs are then cached, allowing future notebooks to bypass this computationally intensive step.

In [3]:
lap_1, lap_2 = preprocess_teammates_data(YEAR, GRAND_PRIX, segment, lap_1, lap_2)

print(f"Drivers names: {lap_1['Driver']}, {lap_2['Driver']}")
print(f"Preprocessed lap times: {lap_1['LapTime']}, {lap_2['LapTime']}")
print(f"Preprocessed lap start times: {lap_1['LapStartTime']}, {lap_2['LapStartTime']}")
print(f"Preprocessed tyre compounds: {lap_1['TyreCompound']}, {lap_2['TyreCompound']}")
print(f"Preprocessed tyre ages: {lap_1['TyreAge']}, {lap_2['TyreAge']}", "\n")
print(f"{lap_1['Driver']} telemetry data info:")
display(lap_1['Telemetry'].info())
print(f"{lap_1['Driver']} telemetry data head:")
display(lap_1['Telemetry'].head())
print(f"{lap_2['Driver']} telemetry data info:")
display(lap_2['Telemetry'].info())
print(f"{lap_2['Driver']} telemetry data head:")
display(lap_2['Telemetry'].head())

Drivers names: ALO, STR
Preprocessed lap times: 88.337, 89.271
Preprocessed lap start times: 0 days 00:31:22.302000, 0 days 00:20:37.637000
Preprocessed tyre compounds: SOFT, SOFT
Preprocessed tyre ages: 2, 2 

ALO telemetry data info:
<class 'fastf1.core.Telemetry'>
RangeIndex: 669 entries, 0 to 668
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Time      669 non-null    float64
 1   Speed     669 non-null    float64
 2   RPM       669 non-null    int64  
 3   nGear     669 non-null    int64  
 4   Throttle  669 non-null    float64
 5   Brake     669 non-null    int64  
 6   Distance  669 non-null    float64
dtypes: float64(4), int64(3)
memory usage: 36.7 KB


None

ALO telemetry data head:


,Time,Speed,RPM,nGear,Throttle,Brake,Distance
0,0.000,277.371429,11407,7,99.0,0,0.103784
1,0.184,280.000000,11549,7,99.0,0,14.417778
2,0.276,280.460000,11600,7,99.0,0,21.579261
3,0.384,281.000000,11660,7,99.0,0,30.028889
4,0.396,281.250000,11664,7,99.0,0,30.973564


STR telemetry data info:
<class 'fastf1.core.Telemetry'>
RangeIndex: 675 entries, 0 to 674
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Time      675 non-null    float64
 1   Speed     675 non-null    float64
 2   RPM       675 non-null    int64  
 3   nGear     675 non-null    int64  
 4   Throttle  675 non-null    float64
 5   Brake     675 non-null    int64  
 6   Distance  675 non-null    float64
dtypes: float64(4), int64(3)
memory usage: 37.0 KB


None

STR telemetry data head:


,Time,Speed,RPM,nGear,Throttle,Brake,Distance
0,0.000,274.938637,11263,7,99.432954,0,1.840340
1,0.381,281.000000,11505,7,99.000000,0,31.679722
2,0.395,281.116666,11512,7,99.038889,0,32.779083
3,0.695,283.616666,11670,7,99.872222,0,56.422592
4,0.741,284.000000,11695,7,100.000000,0,60.079722


## Interpolation

Resample the telemetry data from both drivers onto a unified distance axis. Since drivers cover different distances in slightly different timeframes, this normalization process is critical for enabling a direct, point-by-point performance comparison between teammates.


In [4]:
print(f"\n{lap_1['Driver']} distance vector: \n{lap_1['Telemetry']['Distance'].head()}")
print(f"\n{lap_2['Driver']} distance vector: \n{lap_2['Telemetry']['Distance'].head()}")

n_points = (len(lap_1['Telemetry']['Distance']) + len(lap_2['Telemetry']['Distance'])) // 2
lap_1["Telemetry"], lap_2["Telemetry"] = interpolate_telemetry(lap_1['Telemetry'], lap_2['Telemetry'], lap_1['LapTime'], lap_2['LapTime'], n_points=n_points)
    
print(f"\n{lap_1['Driver']} interpolated data info:")
display(lap_1['Telemetry'].info())
print(f"\n{lap_2['Driver']} interpolated data info:")
display(lap_2['Telemetry'].info())

print(f"\n{lap_1['Driver']} interpolated data description:")
display(lap_1['Telemetry'].describe())
print(f"\n{lap_2['Driver']} interpolated data description:")
display(lap_2['Telemetry'].describe())

print(f"\n{lap_1['Driver']} interpolated data head:")
display(lap_1['Telemetry'].head())
print(f"\n{lap_2['Driver']} interpolated data head:")
display(lap_2['Telemetry'].head())


ALO distance vector: 
0     0.103784
1    14.417778
2    21.579261
3    30.028889
4    30.973564
Name: Distance, dtype: float64

STR distance vector: 
0     1.840340
1    31.679722
2    32.779083
3    56.422592
4    60.079722
Name: Distance, dtype: float64

Interpolation check:
Total distance driver 1: 5761.551 m
Total distance driver 2: 5780.660 m
Difference: 19.109 m
Percentage difference: 0.331%

Time 1 (interpolated): 88.3370  |  LapTime 1: 88.3370
Time 2 (interpolated): 89.2710  |  LapTime 2: 89.2710
Delta (calculated): -0.9340
Delta (expected):   -0.9340
OK: final delta matches the official lap time gap.

ALO interpolated data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Distance  672 non-null    float64
 1   Speed     672 non-null    float64
 2   RPM       672 non-null    int64  
 3   Throttle  672 non-null    float64
 4   nGear    

None


STR interpolated data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672 entries, 0 to 671
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Distance  672 non-null    float64
 1   Speed     672 non-null    float64
 2   RPM       672 non-null    int64  
 3   Throttle  672 non-null    float64
 4   nGear     672 non-null    int64  
 5   Brake     672 non-null    int64  
 6   Time      672 non-null    float64
dtypes: float64(4), int64(3)
memory usage: 36.9 KB


None


ALO interpolated data description:


...

,Distance,Speed,RPM,Throttle,nGear,Brake,Time
count,672.000000,672.000000,672.000000,672.000000,672.000000,672.000000,672.000000
mean,2880.775655,249.379126,10776.255952,83.341958,6.501488,0.126488,43.504657
std,1666.934209,52.359051,879.686899,32.417291,1.416319,0.332646,26.262428
min,0.000000,69.502285,5092.000000,0.000000,3.000000,0.000000,0.000000
25%,1440.387827,224.736610,10575.000000,93.055539,6.000000,0.000000,20.209548
50%,2880.775655,258.937962,10862.000000,99.000000,7.000000,0.000000,43.801581
75%,4321.163482,293.171003,11299.000000,99.350539,8.000000,0.000000,67.025412
max,5761.551310,321.872229,12135.000000,100.000000,8.000000,1.000000,89.271000



ALO interpolated data head:


,Distance,Speed,RPM,Throttle,nGear,Brake,Time
0,0.000000,277.371429,11407,99.0,7,0,0.000000
1,8.586515,278.948226,11492,99.0,7,0,0.110376
2,17.173029,280.183643,11569,99.0,7,0,0.220729
3,25.759544,280.733787,11630,99.0,7,0,0.330757
4,34.346058,282.161176,11679,99.0,7,0,0.439736



STR interpolated data head:


,Distance,Speed,RPM,Throttle,nGear,Brake,Time
0,0.000000,274.938637,11263,99.432954,7,0,0.000000
1,8.586515,276.682842,11332,99.308368,7,0,0.109999
2,17.173029,278.427046,11402,99.183782,7,0,0.219999
3,25.759544,280.171250,11471,99.059196,7,0,0.329998
4,34.346058,281.476946,11534,99.158982,7,0,0.439679


## Cache Processed Data

Persist the fully prepared and interpolated datasets to local cache files. This workflow design ensures that downstream exploratory notebooks can immediately load the high-quality, pre-processed data without re-executing the entire retrieval and transformation pipeline.


In [5]:
try:
    save_cache(YEAR, GRAND_PRIX, segment, lap_1, lap_2)
    save_dataframe(lap_1, "Telemetry", YEAR, GRAND_PRIX)
    save_dataframe(lap_2, "Telemetry", YEAR, GRAND_PRIX)
except Exception as e:
    print(f"Failed to save drivers data to cache: {e}")

Saving payload to E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_japanese_grand_prix_Q1_ALO.pkl
Saved: E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_japanese_grand_prix_Q1_ALO.pkl
Saving payload to E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_japanese_grand_prix_Q1_STR.pkl
Saved: E:\Users\joseg\3D Objects\f1-teammate-analysis\cache\2025_japanese_grand_prix_Q1_STR.pkl

Processed data for 2025 Japanese Grand Prix saved to cache.
Driver 1: ALO, Driver 2: STR, Segment: Q1
